<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install pyspark

In [14]:
# ========== INSTALL AND IMPORT LIBRARIES ==========
!pip install pyspark matplotlib seaborn numpy

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, mean, stddev, min as spark_min, max as spark_max
from pyspark.sql.types import *

# Set style for better visualizations
sns.set_style("whitegrid")
plt.style.use('seaborn-v0_8-darkgrid')

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DataAnalysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print("Libraries imported successfully!")
print(f"Spark version: {spark.version}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")

Libraries imported successfully!
Spark version: 4.0.2
Matplotlib version: 3.10.0
Seaborn version: 0.13.2


In [15]:
# ========== INITIALIZE SPARK AND LOAD DATA ==========
print("="*60)
print("STEP 1: INITIALIZE SPARK AND LOAD DATA")
print("="*60)

# Create Spark Session
spark = SparkSession.builder \
    .appName("EDA_Crop_Yield_Analysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("✓ Spark Session Created")

# Define schema
schema = StructType([
    StructField("Region", StringType(), True),
    StructField("Soil_Type", StringType(), True),
    StructField("Crop", StringType(), True),
    StructField("Rainfall_mm", DoubleType(), True),
    StructField("Temperature_Celsius", DoubleType(), True),
    StructField("Fertilizer_Used", BooleanType(), True),
    StructField("Irrigation_Used", BooleanType(), True),
    StructField("Weather_Condition", StringType(), True),
    StructField("Days_to_Harvest", IntegerType(), True),
    StructField("Yield_tons_per_hectare", DoubleType(), True)
])

# Load data
df = spark.read.csv("/content/crop_yield.csv", header=True, schema=schema)

# Basic data inspection
print("\n--- DATA INSPECTION ---")
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

print("\n--- SCHEMA ---")
df.printSchema()

print("\n--- FIRST 5 ROWS df.show(5) ---")
df.show(5)

STEP 1: INITIALIZE SPARK AND LOAD DATA
✓ Spark Session Created

--- DATA INSPECTION ---
Total rows: 1,000,000
Total columns: 10

--- SCHEMA ---
root
 |-- Region: string (nullable = true)
 |-- Soil_Type: string (nullable = true)
 |-- Crop: string (nullable = true)
 |-- Rainfall_mm: double (nullable = true)
 |-- Temperature_Celsius: double (nullable = true)
 |-- Fertilizer_Used: boolean (nullable = true)
 |-- Irrigation_Used: boolean (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Days_to_Harvest: integer (nullable = true)
 |-- Yield_tons_per_hectare: double (nullable = true)


--- FIRST 5 ROWS df.show(5) ---
+------+---------+-------+-----------------+-------------------+---------------+---------------+-----------------+---------------+----------------------+
|Region|Soil_Type|   Crop|      Rainfall_mm|Temperature_Celsius|Fertilizer_Used|Irrigation_Used|Weather_Condition|Days_to_Harvest|Yield_tons_per_hectare|
+------+---------+-------+-----------------+---------

In [16]:
# ========== DATA CLEANING ==========
print("="*60)
print("DATA CLEANING - BEFORE AND AFTER")
print("="*60)

# BEFORE CLEANING
print("\n--- BEFORE CLEANING ---")
total_rows = df.count()
print(f"Total rows: {total_rows:,}")

print("\nMissing values count:")
for column in df.columns:
    null_count = df.filter(col(column).isNull()).count()
    if null_count > 0:
        print(f"  {column}: {null_count:,}")

# Show rows with ANY missing values
print("\nRows with ANY missing values:")
null_condition = None
for column in df.columns:
    if null_condition is None:
        null_condition = col(column).isNull()
    else:
        null_condition = null_condition | col(column).isNull()

df.filter(null_condition).show(10)

# Count rows with ANY missing values
rows_with_nulls = df.filter(null_condition).count()
print(f"\nTotal rows with missing values: {rows_with_nulls:,}")

# AFTER CLEANING
print("\n--- AFTER CLEANING ---")
df_clean = df.dropna()

print(f"Rows after dropna(): {df_clean.count():,}")
print(f"Rows removed: {total_rows - df_clean.count():,}")

print("\nMissing values after dropna():")
for column in df_clean.columns:
    null_count = df_clean.filter(col(column).isNull()).count()
    if null_count > 0:
        print(f"  {column}: {null_count:,}")

if df_clean.count() == total_rows:
    print("\n✓ No rows were removed - no missing values found in any column!")
else:
    print(f"\n✓ Removed {total_rows - df_clean.count():,} rows with missing values")

df = df_clean

DATA CLEANING - BEFORE AND AFTER

--- BEFORE CLEANING ---
Total rows: 1,000,000

Missing values count:

Rows with ANY missing values:
+------+---------+----+-----------+-------------------+---------------+---------------+-----------------+---------------+----------------------+
|Region|Soil_Type|Crop|Rainfall_mm|Temperature_Celsius|Fertilizer_Used|Irrigation_Used|Weather_Condition|Days_to_Harvest|Yield_tons_per_hectare|
+------+---------+----+-----------+-------------------+---------------+---------------+-----------------+---------------+----------------------+
+------+---------+----+-----------+-------------------+---------------+---------------+-----------------+---------------+----------------------+


Total rows with missing values: 0

--- AFTER CLEANING ---
Rows after dropna(): 1,000,000
Rows removed: 0

Missing values after dropna():

✓ No rows were removed - no missing values found in any column!


In [17]:
# ========== REMOVE DUPLICATES ==========
print("="*60)
print("REMOVING DUPLICATES - BEFORE AND AFTER")
print("="*60)

# Get counts as integers
count_before = df.count()
print(f"\nBefore: {count_before:,} rows")

# Remove duplicates
df_clean = df.distinct()

# Get counts after
count_after = df_clean.count()
print(f"After: {count_after:,} rows")
print(f"Removed: {count_before - count_after:,} duplicates")

# Assign back to df
df = df_clean

REMOVING DUPLICATES - BEFORE AND AFTER

Before: 1,000,000 rows
After: 1,000,000 rows
Removed: 0 duplicates


In [18]:
# ========== FINAL VERIFICATION ==========
print("="*60)
print("FINAL CLEAN DATA")
print("="*60)

print(f"\nRows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
print(f"Missing values: {df.select([col(c).isNull().alias(c) for c in df.columns]).where(col("Rainfall_mm") == True).count()}")

print("\nFirst 5 rows:")
df.show(5)

FINAL CLEAN DATA

Rows: 1,000,000
Columns: 10
Missing values: 0

First 5 rows:
+------+---------+-------+------------------+-------------------+---------------+---------------+-----------------+---------------+----------------------+
|Region|Soil_Type|   Crop|       Rainfall_mm|Temperature_Celsius|Fertilizer_Used|Irrigation_Used|Weather_Condition|Days_to_Harvest|Yield_tons_per_hectare|
+------+---------+-------+------------------+-------------------+---------------+---------------+-----------------+---------------+----------------------+
|  West|     Clay|  Wheat|107.55870219265138| 27.429970341193737|          false|          false|            Sunny|            138|    1.1510530575473923|
|  East|    Peaty|Soybean|  825.632804798505| 29.738417997388506|          false|           true|           Cloudy|             70|     6.895785071421378|
|  West|     Loam| Barley| 370.2483963822427| 18.310242352944513|          false|          false|            Sunny|            147|    1.899373625

In [19]:
print("Hello world!")

Hello world!
